In [ ]:
! pip install -q torchattacks 

In [2]:
import os
import json
import torch
import torchvision
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm.auto import tqdm
import random
import gc
from torch.amp import autocast, GradScaler
import torch.nn as nn
from scipy.stats import norm
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import math
import numpy as np
from torchvision.datasets import CIFAR10
import glob
from torchvision import transforms
import torchattacks
import torch.nn.functional as F
import pandas as pd
import copy

%config InlineBackend.figure_format = 'retina'

In [3]:
class JNDModel:
    
    def __init__(self, L_min, L_max, t = 1, phi_d = 1e-3, p = 0.75):
        self.L_min = L_min
        self.L_max = L_max
        self.t = t
        self.phi_d = phi_d
        self.p = p
        self.L0 = 1e-6

    def _K(self, lambd, b1 = 0.98, b2 = 0.1, lambd1 = 2e-2):
        lambd = np.asarray(lambd, dtype = np.float64)
        return np.where(lambd <= 1, b1 * (1 + lambd1 / lambd), lambd ** b2)
    
    def _A(self, La, a1 = 4.8e6, a2 = 7.1e3, a3 = 4.5e-4): 
        return 1 + (a1 * La) ** 0.5 + a2 * La * (1 + (a3 * La) ** 0.5)
    
    def _Lambda(self, lambd):
        return lambd * self._K(lambd)
    
    def _eta(self, La, L3 = 1e6, L4 = 1, a4 = 600):
        return 1 + L3 / (1 + a4 * La * (L4 + La))
    
    def _C(self, La, t):
        tao_min = 0.05
        T = t / (tao_min * self._eta(La))
        return 1 + 1 / T 
    
    def _func_phi_1(self, La, c2 = 0.25, L1 = 10, L2 = 2.6e-5):
        return (1 + L1 / (L2 + La)) ** c2
    
    def _S(self, La, theta_1 = 6.1, a_d = 2, inf_phi_1 = 1):
        phi_0_min = 0.5 * (1 / 60) * (np.pi / 180)
        phi_1 = self._func_phi_1(La)
        phi_0 = phi_1 * phi_0_min / inf_phi_1
        
        return (theta_1 * phi_0 / self.phi_d + 1.0) ** a_d
    
    def _l(self, La):
        return self.L0 * self._A(La)
    
    def _P(self, p, p_ref = 0.75):
        return abs(norm.ppf(p) / norm.ppf(p_ref))

    def _D(self, L, La):
        lambd = L / (10 ** 2 * self._l(La))
        return self.L0 * self._A(La) * self._C(La, self.t) * self._Lambda(lambd) * self._S(La, self.phi_d) * self._P(self.p)

    def find_La(self, L_matrix):
        start_La = np.mean(L_matrix)
        EPS = 1e-12
        logL = np.log(np.maximum(L_matrix, EPS))
    
        # Границы можно расширить, если адаптация может лежать вне диапазона яркостей картинки.
        log_La_min = float(logL.min())
        log_La_max = float(logL.max())
    
        def S_score(L_matrix, start_La):
            numerator = self._D(L_matrix, start_La)
            denominator = self._D(L_matrix, L_matrix)
            return float(np.mean(np.log(numerator / denominator)))
        
        def objective_log_La(log_La):
            return S_score(L_matrix, math.exp(float(log_La)))
        
        result = minimize_scalar(
            objective_log_La,
            bounds=(log_La_min, log_La_max),
            method='bounded',
            options={'xatol': 1e-6},
        )
        
        log_La_star = float(result.x)
        self.La = math.exp(log_La_star)
        
        return self.La

    def build_level_boundaries(self):
        L_right_bounds = []
        L_curr = self.La
        while L_curr < self.L_max:
            L_curr += self._D(L_curr, self.La)
            L_right_bounds.append(min(L_curr, self.L_max))

        L_left_bounds = []
        L_curr = self.La
        while L_curr > self.L0:
            L_curr -= self._D(L_curr, self.La)
            L_left_bounds.append(max(L_curr, self.L0))

        self.len_left = len(L_left_bounds)

        L_left_bounds.reverse()

        self.bounds = np.array(L_left_bounds + [self.La] + L_right_bounds)

        return self.bounds
    
    def L_to_k(self, L):
        return np.searchsorted(self.bounds, L, side = 'right') - 1 - self.len_left
    
    def k_to_L(self, k):
        return self.bounds[k + self.len_left]

In [4]:
class ImageConverter:

    def __init__(self):
        self.M = np.array([
            [0.4124564,  0.3575761,  0.1804375],
            [0.2126729,  0.7151522,  0.0721750],
            [0.0193339,  0.1191920,  0.9503041]
        ])

        """
        M_inv: обратная матрица цветового преобразования (XYZ->LinRGB)
        """

        self.M_inv = np.array([
            [ 3.1338561, -1.6168667, -0.4906146],
            [-0.9787684,  1.9161415,  0.0334540],
            [ 0.0719453, -0.2289914,  1.4052427]
        ])
        
    def read_img(self, image):
        V = cv2.imread(image)
        V = cv2.cvtColor(V, cv2.COLOR_BGR2RGB)
        V = V.astype(np.float64)
        return V
    
    def normalize_image(self, image):
        if image.max() > 1:
            image = image / 255.0
        
        return image
    
    def Inverse_sRGB_Companding(self, image):
        
        mask = image <= 0.04045
        image[mask] = image[mask] / 12.92
        image[~mask] = ((image[~mask] + 0.055) / 1.055) ** 2.4
    
        return image
    
    def Linear_RGB_to_XYZ(self, image):
        XYZ_image = image.reshape(-1, 3)
        XYZ_image =  (XYZ_image @ self.M.T).reshape(image.shape)
    
        # канал Y = яркость
        # print("Яркость изображения linRGB -> XYZ формате")
        # plt.imshow(XYZ_image[:, :, 1], cmap="grey")
        # plt.colorbar()
        # plt.axis("off")
        # plt.show()
        
        return XYZ_image
    
    def XYZ_to_xyY(self, XYZ_image, L_max=617.0, coord_white=(0.3127, 0.3290)):
        """
        XYZ_image: входное изображениe формата XYZ
        coord_white: координаты x, y для белого цвета (D65 CIE)
        """
        # сумма каналов X + Y + Z для каждого пикселя
        sum_xyz = XYZ_image.sum(axis=-1, keepdims=True)
        # eps, чтобы избежать деления на 0
        eps = 1e-10 
        
        # нормализация координат xy
        xy = XYZ_image[..., :2] / (sum_xyz + eps)
        
        # маска черного цвета (сумма каналов ~0)
        is_black = (sum_xyz.squeeze(-1) < eps)
        
        # если пиксель черный, установка координат белой точки
        if np.any(is_black):
            xy[is_black] = coord_white
            
        # Y канал не меняется
        Y = XYZ_image[..., 1]
        
        # изображение xyY
        xyY_image = np.stack([xy[..., 0], xy[..., 1], Y], axis=-1)
    
        # нормировка яркости на L_max
        L_norm = np.clip(Y / L_max, 0, 1)
    
        # Изображение xyL
        xyL_norm_image = np.stack([xy[..., 0], xy[..., 1], L_norm], axis=-1)
        
        return xyL_norm_image

    def xyL_to_sRGB(self, xyL_image, L_max=617.0):
     
        x = xyL_image[:, :, 0]
        y = xyL_image[:, :, 1]
        L = xyL_image[:, :, 2]
        
        # восстановление абсолютной яркости
        Y = L * L_max
        
        # восстановление каналов
        eps = 1e-10
        factor = Y / (y + eps) 
        
        X = x * factor
        Z = (1.0 - x - y) * factor
        
        XYZ_img = np.stack([X, Y, Z], axis=-1)
        
        # XYZ -> Linear RGB
        lin_RGB = XYZ_img @ self.M_inv
        
        # Linear RGB -> sRGB
        mask_low = lin_RGB <= 0.0031308
        srgb = np.where(
            mask_low, 
            12.92 * lin_RGB, 
            1.055 * np.power(np.clip(lin_RGB, 0, None), 1.0/2.4) - 0.055
        )
        
        return np.clip(srgb, 0, 1)

    def RGB_to_xyL(self, img, L_min, L_max):
        
        """
        Переход из sRGB в физическую яркость: sGRB -> linRGB -> XYZ -> L -> xLy (<-> sRGB)
        L_min - минимальная яркость дисплея
        L_max - максимальная яркость дисплея
        """
        
        V = self.read_img(img)
    
        # [0, 1]
        norm_img = self.normalize_image(V)
    
        # sRGB -> linRGB
        linRGBimg = self.Inverse_sRGB_Companding(norm_img)
    
        #linRGB -> XYZ
        XYZ_image = self.Linear_RGB_to_XYZ(linRGBimg)
    
        xyL_image = self.XYZ_to_xyY(XYZ_image, L_max)

        return xyL_image

    def RGB_to_XL_normZ(self, img, L_min, L_max):
        
        V = self.read_img(img)
    
        norm_img = self.normalize_image(V)
    
        linRGBimg = self.Inverse_sRGB_Companding(norm_img)
    
        XYZ_image = self.Linear_RGB_to_XYZ(linRGBimg)
    
        xyL_image = self.XYZ_to_xyY(XYZ_image, L_max)

        X_absolute = XYZ_image[..., 0]
        
        Z_absolute = XYZ_image[..., 2]
        
        L_norm = xyL_image[..., 2]

        XL_normZ_image = np.stack([X_absolute, L_norm, Z_absolute], axis=-1)

        return XL_normZ_image

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [6]:
random.seed(42)
torch.manual_seed(42)

data_path = '/kaggle/input/datasets/gazu468/cifar10-classification-image/cifar10'

train_path = f'{data_path}/train'
val_path = f'{data_path}/test'

In [7]:
jnd_train_path = '/kaggle/working/cifar10_train_jnd'
jnd_val_path = '/kaggle/working/cifar10_val_jnd'

if not (os.path.exists(jnd_train_path) and os.path.exists(jnd_val_path)):
    
    from multiprocessing import Pool
    
    L_MIN = 0.1
    L_MAX = 300.0
    NUM_WORKERS = 4  
    
    converter = None
    jnd_model = None
    
    def init_worker():
        global converter, jnd_model
        import sys
        import numpy as np
        import math
        from scipy.stats import norm
        from scipy.optimize import minimize_scalar
        
        sys.modules['np'] = np
        globals()['np'] = np
        sys.modules['math'] = math
        globals()['math'] = math
        globals()['norm'] = norm
        globals()['minimize_scalar'] = minimize_scalar
        converter = ImageConverter()
        jnd_model = JNDModel(L_MIN, L_MAX)
    
    def process_single_image(task):
        img_path, save_path = task = task
        
        try:
            if os.path.exists(save_path):
                return True
                
            xyL_image = converter.RGB_to_xyL(img_path, L_MIN, L_MAX)
            
            x_coord = xyL_image[..., 0]
            y_coord = xyL_image[..., 1]
            L_norm = xyL_image[..., 2]
            
            L_physical = L_MIN + (L_MAX - L_MIN) * L_norm
            
            jnd_model.find_La(L_physical)
            jnd_model.build_level_boundaries()
            k_map = jnd_model.L_to_k(L_physical) 
            
            xyL_jnd_tensor = np.stack([x_coord, y_coord, k_map], axis=-1)
            
            np.save(save_path, xyL_jnd_tensor.astype(np.float32))
            
            del xyL_image, L_norm, L_physical, k_map, xyL_jnd_tensor
            
            return True
            
        except Exception as e:
            print(f"Ошибка {e}")
            return False
    
    def save_jnd_folder(input_dir, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        classes = sorted(os.listdir(input_dir))
        tasks = []
        
        for cls_name in classes:
            os.makedirs(os.path.join(output_dir, cls_name), exist_ok=True)
            cls_folder = os.path.join(input_dir, cls_name)
            file_paths = glob.glob(os.path.join(cls_folder, "*.png"))
            
            for path in file_paths:
                file_name = os.path.basename(path)
                save_name = file_name.replace(".png", ".npy")
                save_path = os.path.join(output_dir, cls_name, save_name)
                tasks.append((path, save_path))
        
        with Pool(processes=NUM_WORKERS, initializer=init_worker) as pool:
            with tqdm(total=len(tasks), desc=f"Создание {os.path.basename(output_dir)}") as pbar:
                for _ in pool.imap_unordered(process_single_image, tasks):
                    pbar.update(1)

    save_jnd_folder(train_path, jnd_train_path)
    
    save_jnd_folder(val_path, jnd_val_path)
    
    print('Done!')
    

Создание cifar10_train_jnd:   0%|          | 0/50000 [00:00<?, ?it/s]

Создание cifar10_val_jnd:   0%|          | 0/10000 [00:00<?, ?it/s]

Done!


In [8]:
class ProxyCifar10Dataset(Dataset):
    def __init__(self, rgb_dir, jnd_dir, is_train, layers=None):
        self.layers = layers
        self.is_train = is_train
        self.rgb_dir = rgb_dir
        self.jnd_dir = jnd_dir
        self.classes = sorted(os.listdir(rgb_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.samples = []
        
        for cls_name in self.classes:
            rgb_cls_folder = os.path.join(rgb_dir, cls_name)
            jnd_cls_folder = os.path.join(jnd_dir, cls_name)
            
            rgb_files = sorted(glob.glob(os.path.join(rgb_cls_folder, "*.png")))
            for rgb_path in rgb_files:
                file_name = os.path.basename(rgb_path)
                jnd_name = file_name.replace(".png", ".npy")
                jnd_path = os.path.join(jnd_cls_folder, jnd_name)
                
                if os.path.exists(jnd_path):
                    self.samples.append((rgb_path, jnd_path, self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def apply_augmentations(self, rgb_tensor, jnd_tensor):
        if not self.is_train:
            return rgb_tensor, jnd_tensor

        if torch.rand(1) > 0.5:
            rgb_tensor = TF.hflip(rgb_tensor)
            jnd_tensor = TF.hflip(jnd_tensor)
            
        angle = random.uniform(-10, 10)
        rgb_tensor = TF.rotate(rgb_tensor, angle, interpolation=TF.InterpolationMode.NEAREST)
        jnd_tensor = TF.rotate(jnd_tensor, angle, interpolation=TF.InterpolationMode.NEAREST)
        
        rgb_tensor = TF.pad(rgb_tensor, padding=4, padding_mode='reflect')
        jnd_tensor = TF.pad(jnd_tensor, padding=4, padding_mode='reflect')
        
        i, j, h, w = transforms.RandomCrop.get_params(rgb_tensor, output_size=(32, 32))
        rgb_tensor = TF.crop(rgb_tensor, i, j, h, w)
        jnd_tensor = TF.crop(jnd_tensor, i, j, h, w)
        
        return rgb_tensor, jnd_tensor
    
    def __getitem__(self, idx):
        rgb_path, jnd_path, label = self.samples[idx]
        
        rgb_img = Image.open(rgb_path).convert('RGB')
        rgb_tensor = TF.to_tensor(rgb_img)
        
        jnd_matrix = np.load(jnd_path).astype(np.float32)
        if self.layers == 'x_y_jnd':
            jnd_tensor = torch.from_numpy(jnd_matrix).permute(2, 0, 1)
        elif self.layers == 'jnd_jnd_jnd':
            k_map = jnd_matrix[:, :, 2]
            jnd_3channel = np.stack([k_map, k_map, k_map], axis=-1)
            jnd_tensor = torch.from_numpy(jnd_3channel).permute(2, 0, 1)
        elif self.layers == 'X_jnd_Z':
            jnd_tensor = torch.from_numpy(jnd_matrix).permute(2, 0, 1)
        else:
            raise ValueError(f"Unknown layers mode: {self.layers}")

        rgb_tensor, jnd_tensor = self.apply_augmentations(rgb_tensor, jnd_tensor)

        return rgb_tensor, jnd_tensor, torch.tensor(label, dtype=torch.long)

In [9]:
def train_proxy(parent_model, proxy_model, optimizer, scheduler, epochs, 
                             train_loader, val_loader, model_name, temperature=3.0, device='cuda'):
    losses_history = []
    fidelity_history = []
    best_fidelity = 0.0
    
    parent_model.eval()
    for param in parent_model.parameters():
        param.requires_grad = False
        
    criterion_kl = nn.KLDivLoss(reduction='batchmean')
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(epochs):
        proxy_model.train()
        epoch_train_loss = 0.0
        
        for rgb_images, jnd_tensors, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train Proxy]'):
            rgb_images = rgb_images.to(device)
            jnd_tensors = jnd_tensors.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                parent_logits = parent_model(jnd_tensors)

            with torch.amp.autocast('cuda'):
                proxy_logits = proxy_model(rgb_images)
                soft_proxy = F.log_softmax(proxy_logits / temperature, dim=1)
                soft_parent = F.softmax(parent_logits / temperature, dim=1)
                loss = criterion_kl(soft_proxy, soft_parent) * (temperature ** 2)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        losses_history.append(avg_train_loss)

        proxy_model.eval()
        matched_predictions = 0
        total_samples = 0
        
        with torch.no_grad():
            for rgb_images, jnd_tensors, _ in tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val Fidelity]'):
                rgb_images = rgb_images.to(device)
                jnd_tensors = jnd_tensors.to(device)
                
                parent_outputs = parent_model(jnd_tensors)
                _, parent_preds = torch.max(parent_outputs, dim=1)
                
                proxy_outputs = proxy_model(rgb_images)
                _, proxy_preds = torch.max(proxy_outputs, dim=1)
                
                matched_predictions += torch.sum(proxy_preds == parent_preds).item()
                total_samples += rgb_images.size(0)
        
        epoch_fidelity = matched_predictions / total_samples
        fidelity_history.append(epoch_fidelity)

        scheduler.step()

        if epoch_fidelity > best_fidelity:
            best_fidelity = epoch_fidelity
            torch.save(proxy_model.state_dict(), f'{model_name}.pth')

        print(f'Epoch: {epoch + 1}/{epochs}. Train loss (KL): {avg_train_loss:.4f}. Fidelity: {epoch_fidelity*100:.2f}%')

    return losses_history, fidelity_history

In [10]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.LeakyReLU(0.1, inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.downsample = downsample
    
    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.LeakyReLU(0.1, inplace=True)

        self.layer1 = self._make_layer(64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(512, num_blocks=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        layers = []
        layers.append(BasicBlock(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels
        
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x

In [32]:
parent_model = ResNet18().to(device)

In [33]:
parent_model.load_state_dict(torch.load('/kaggle/input/models/mishasavinov/4-jnd-models/pytorch/default/1/jnd_models_weights/model_3jnd_weight.pth'))

<All keys matched successfully>

In [34]:
train_dataset = ProxyCifar10Dataset(rgb_dir=train_path, jnd_dir=jnd_train_path, is_train=True, layers='jnd_jnd_jnd')
val_dataset = ProxyCifar10Dataset(rgb_dir=val_path, jnd_dir=jnd_val_path, is_train=False, layers='jnd_jnd_jnd')

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

In [35]:
proxy_model = ResNet18().to(device)

optimizer = torch.optim.AdamW(proxy_model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

In [36]:
losses, fidelities = train_proxy(
    parent_model=parent_model,
    proxy_model=proxy_model,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=100,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name="proxy_model",
    temperature=3.0,
    device=device
)

Epoch 1/100 [Train Proxy]:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 1/100 [Val Fidelity]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch: 1/100. Train loss (KL): 8.4668. Fidelity: 54.18%


Epoch 2/100 [Train Proxy]:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 2/100 [Val Fidelity]:   0%|          | 0/79 [00:00<?, ?it/s]

...

Epoch 99/100 [Val Fidelity]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch: 99/100. Train loss (KL): 0.9474. Fidelity: 77.90%


Epoch 100/100 [Train Proxy]:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch 100/100 [Val Fidelity]:   0%|          | 0/79 [00:00<?, ?it/s]

Epoch: 100/100. Train loss (KL): 0.9400. Fidelity: 77.96%


In [38]:
class Cifar10Dataset(Dataset):
    def __init__(self, data_dir, is_jnd, is_train, layers = None):
        self.layers = layers
        self.is_train = is_train
        self.data_dir = data_dir
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.is_jnd = is_jnd
        self.samples = []
        
        extension = "*.npy" if is_jnd else "*.png"
        
        for cls_name in self.classes:
            cls_folder = os.path.join(data_dir, cls_name)
            
            file_paths = glob.glob(os.path.join(cls_folder, extension))
            
            class_idx = self.class_to_idx[cls_name]
            
            for path in file_paths:
                self.samples.append((path, class_idx))

    def __len__(self):
        return len(self.samples)

    def apply_augmentations(self, image):
        if not self.is_train:
            return image

        if torch.rand(1) > 0.5:
            image = TF.hflip(image)
            
        angle = random.uniform(-10, 10)
        
        image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.NEAREST)
        
        image = TF.pad(image, padding=4, padding_mode='reflect')
        
        i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(32, 32))
        
        image = TF.crop(image, i, j, h, w)
        
        return image
    
    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        if self.is_jnd:
            jnd_matrix = np.load(file_path).astype(np.float32)
            if self.layers == 'x_y_jnd':
                image = torch.from_numpy(jnd_matrix).permute(2, 0, 1)
            elif self.layers == 'jnd_jnd_jnd':
                k_map = jnd_matrix[:, :, 2]
                jnd_3channel = np.stack([k_map, k_map, k_map], axis=-1)
                image = torch.from_numpy(jnd_3channel).permute(2, 0, 1)
            elif self.layers == 'X_jnd_Z':
                image = torch.from_numpy(jnd_matrix).permute(2, 0, 1)
        else:
            image = Image.open(file_path).convert('RGB')
            image = TF.to_tensor(image)

        image = self.apply_augmentations(image)

        return image, torch.tensor(label, dtype=torch.long)

In [39]:
val_dataset_rgb = Cifar10Dataset(val_path, is_jnd=False, is_train=False)

indices = list(random.sample(range(10000), 1000))
subset_dataset = Subset(val_dataset_rgb, indices)

val_loader_1000_rgb = DataLoader(subset_dataset, batch_size=16, shuffle=False)

In [40]:
def rgb_tensor_to_tensor(rgb_tensor, converter, jnd_model, L_MIN, L_MAX, layers=None):
    """
    Принимает тензор RGB [B, 3, H, W] в диапазоне [0, 1].
    Конвертирует его в нужный JND-формат в зависимости от значения layers:
    - 'x_y_jnd':     [x, y, k_map]
    - 'jnd_jnd_jnd': [k_map, k_map, k_map]
    - 'X_jnd_Z':     [X, k_map, Z]
    Возвращает тензор [B, 3, H, W]
    """

    rgb_np = rgb_tensor.permute(0, 2, 3, 1).cpu().numpy().astype(np.float64)
    
    jnd_batch = []
    for i in range(rgb_np.shape[0]):
        img = rgb_np[i].copy()

        norm_img = converter.normalize_image(img)
        linRGBimg = converter.Inverse_sRGB_Companding(norm_img)
        XYZ_image = converter.Linear_RGB_to_XYZ(linRGBimg)
        xLy_image = converter.XYZ_to_xyY(XYZ_image, L_MAX)
        
        x_coord = xLy_image[..., 0]
        y_coord = xLy_image[..., 1]
        L_norm = xLy_image[..., 2]
        
        L_physical = L_MIN + (L_MAX - L_MIN) * L_norm
        
        jnd_model.find_La(L_physical)
        jnd_model.build_level_boundaries()
        k_map = jnd_model.L_to_k(L_physical)
        
        if layers == 'x_y_jnd':
            output_matrix = np.stack([x_coord, y_coord, k_map], axis=-1)
            
        elif layers == 'jnd_jnd_jnd':
            output_matrix = np.stack([k_map, k_map, k_map], axis=-1)
            
        elif layers == 'X_jnd_Z':
            eps = 1e-10
            
            factor = L_physical / (y_coord + eps)
            
            X_absolute = x_coord * factor
            Z_absolute = (1.0 - x_coord - y_coord) * factor
            
            output_matrix = np.stack([X_absolute, k_map, Z_absolute], axis=-1)
            
        else:
            raise ValueError(f"Неизвестный режим слоев: {layers}")
            
        jnd_batch.append(output_matrix)
        
    jnd_batch_np = np.stack(jnd_batch, axis=0)
    
    jnd_tensor = torch.from_numpy(jnd_batch_np.astype(np.float32)).permute(0, 3, 1, 2).to(rgb_tensor.device)
    return jnd_tensor

In [41]:
def evaluate_single_model_robustness(model, attack_model, attack_class, loader, converter, jnd_model, L_MIN, L_MAX, layers="rgb", **kwargs):
    model.eval()
    attack_model.eval()
    
    total_clean_correct, total_adv_correct = 0, 0
    total_clean_conf, total_adv_conf = 0.0, 0.0
    total_images = 0
    
    attack = attack_class(attack_model, **kwargs)
    device = next(model.parameters()).device
    
    for images, labels in tqdm(loader, desc=f"Атака {attack_class.__name__}", leave=False):
        images, labels = images.to(device), labels.to(device)
        adv_images_rgb = attack(images, labels)
        
        def prepare_inputs(input_rgb):
            if layers == "rgb":
                return input_rgb
            
            layer_mapping = {
                "jnd_xyk": "x_y_jnd",
                "jnd_kkk": "jnd_jnd_jnd",
                "jnd_xkz": "X_jnd_Z"
            }
            target_layer = layer_mapping.get(layers, layers)
            return rgb_tensor_to_tensor(input_rgb, converter, jnd_model, L_MIN, L_MAX, layers=target_layer)

        clean_inputs = prepare_inputs(images)
        adv_inputs = prepare_inputs(adv_images_rgb)
        
        with torch.no_grad():
            outputs_clean = model(clean_inputs)
            _, predicted_clean = torch.max(outputs_clean.data, 1)
            total_clean_correct += (predicted_clean == labels).sum().item()
            probs_clean = F.softmax(outputs_clean, dim=1)
            total_clean_conf += probs_clean.gather(1, labels.view(-1, 1)).squeeze().sum().item()
            
            outputs_adv = model(adv_inputs)
            _, predicted_adv = torch.max(outputs_adv.data, 1)
            total_adv_correct += (predicted_adv == labels).sum().item()
            probs_adv = F.softmax(outputs_adv, dim=1)
            total_adv_conf += probs_adv.gather(1, labels.view(-1, 1)).squeeze().sum().item()
        
        total_images += len(labels)
        
    return {
        "Clean Acc": 100 * total_clean_correct / total_images,
        "Adv Acc": 100 * total_adv_correct / total_images,
        "Drop": 100 * (total_clean_correct - total_adv_correct) / total_images,
        "Clean Conf": total_clean_conf / total_images,
        "Adv Conf": total_adv_conf / total_images,
    }

def get_results_for_model(model, attack_model, loader, converter, jnd_model, L_MIN, L_MAX, layers="rgb"):
    data = []
    attacks = [torchattacks.FGSM, torchattacks.BIM, torchattacks.PGD, torchattacks.CW]
    names = ["FGSM", "BIM", "PGD", "CW"]
    params = [
        {"eps": 8/255},
        {"eps": 8/255, "alpha": 2/255, "steps": 10},
        {"eps": 8/255, "alpha": 1/255, "steps": 10, "random_start": True},
        {"c": 1, "kappa": 0, "steps": 50, "lr": 0.01}
    ]
    
    for i, attack_class in tqdm(enumerate(attacks), desc=f'Тестирование режима {layers}', total=4):
        res = evaluate_single_model_robustness(
            model=model,
            attack_model=attack_model,
            attack_class=attack_class,
            loader=loader,
            converter=converter,
            jnd_model=jnd_model,
            L_MIN=L_MIN,
            L_MAX=L_MAX,
            layers=layers,
            **params[i]
        )
        res["Attack"] = names[i]
        data.append(res)
        
    return pd.DataFrame(data).set_index("Attack")

def visualize_transfer_attack_samples(model, attack_model, attack_class, loader, layers="rgb", num_images=10, class_names=None, 
                                     converter=None, jnd_model=None, L_MIN=0.1, L_MAX=300.0, **kwargs):
    if converter is None:
        converter = ImageConverter()
    if jnd_model is None:
        jnd_model = JNDModel(L_MIN, L_MAX)
        
    model.eval()
    attack_model.eval()
    attack = attack_class(attack_model, **kwargs)
    
    images, labels = next(iter(loader))
    device = next(attack_model.parameters()).device
    images, labels = images.to(device), labels.to(device)

    adv_images_rgb = attack(images, labels)
    
    def prepare_inputs(input_rgb):
        if layers == "rgb":
            return input_rgb
        layer_mapping = {
            "jnd_xyk": "x_y_jnd",
            "jnd_kkk": "jnd_jnd_jnd",
            "jnd_xkz": "X_jnd_Z"
        }
        target_layer = layer_mapping.get(layers, layers)
        return rgb_tensor_to_tensor(input_rgb, converter, jnd_model, L_MIN, L_MAX, layers=target_layer)

    clean_inputs = prepare_inputs(images)
    adv_inputs = prepare_inputs(adv_images_rgb)
    
    with torch.no_grad():
        outputs_clean = model(clean_inputs)
        _, preds_clean = torch.max(outputs_clean, 1)
        
        outputs_adv = model(adv_inputs)
        _, preds_adv = torch.max(outputs_adv, 1)
    
    images_np = images.permute(0, 2, 3, 1).cpu().numpy()
    adv_images_np = adv_images_rgb.permute(0, 2, 3, 1).cpu().numpy()
    labels = labels.cpu().numpy()
    preds_clean = preds_clean.cpu().numpy()
    preds_adv = preds_adv.cpu().numpy()
    
    num_images = min(num_images, len(images_np))
    fig, axes = plt.subplots(2, num_images, figsize=(num_images * 2.8, 10))
    
    def get_physical_luminance(img_rgb):
        img_input = np.clip(img_rgb, 0, 1).astype(np.float64) 
        linRGBimg = converter.Inverse_sRGB_Companding(img_input)
        XYZ_image = converter.Linear_RGB_to_XYZ(linRGBimg)
        Y_relative = XYZ_image[..., 1]
        return L_MIN + (L_MAX - L_MIN) * Y_relative

    for i in range(num_images):
        true_label = class_names[labels[i]] if class_names else str(labels[i])
        pred_clean_label = class_names[preds_clean[i]] if class_names else str(preds_clean[i])
        pred_adv_label = class_names[preds_adv[i]] if class_names else str(preds_adv[i])
        
        img_clean = images_np[i]
        img_adv = adv_images_np[i]

        L_physical_clean = get_physical_luminance(img_clean)
        La_clean = jnd_model.find_La(L_physical_clean)
        jnd_model.build_level_boundaries() 
        k_map_clean = jnd_model.L_to_k(L_physical_clean)

        L_physical_adv = get_physical_luminance(img_adv)
        La_adv = jnd_model.find_La(L_physical_adv)
        jnd_model.build_level_boundaries()
        k_map_adv = jnd_model.L_to_k(L_physical_adv)
        
        min_k_clean, max_k_clean = np.min(k_map_clean), np.max(k_map_clean)
        min_k_adv, max_k_adv = np.min(k_map_adv), np.max(k_map_adv)

        ax = axes[0, i]
        ax.imshow(np.clip(img_clean, 0, 1))
        ax.axis('off')
        if i == 0: ax.set_title("Clean RGB", loc='left', fontsize=18, pad=10)
        clean_color = 'green' if preds_clean[i] == labels[i] else 'red'
        ax.text(0.5, -0.05, f"True: {true_label}\nParent Pred: {pred_clean_label}\nLa: {La_clean:.1f}\nk: {int(min_k_clean)}/{int(max_k_clean)}", 
                transform=ax.transAxes, ha="center", va="top", color=clean_color, fontsize=11)
            
        ax = axes[1, i]
        ax.imshow(np.clip(img_adv, 0, 1))
        ax.axis('off')
        if i == 0: ax.set_title("Adv RGB (from Proxy)", loc='left', fontsize=18, pad=10)
  
        adv_color = 'red' if preds_adv[i] != labels[i] else 'green'
        ax.text(0.5, -0.05, f"True: {true_label}\nParent Pred: {pred_adv_label}\nLa: {La_adv:.1f}\nk: {int(min_k_adv)}/{int(max_k_adv)}", 
                transform=ax.transAxes, ha="center", va="top", color=adv_color, fontsize=11)
         
    plt.tight_layout() 
    plt.show()

In [45]:
L_MIN = 0.1
L_MAX = 300.0
converter = ImageConverter()
jnd_model = JNDModel(L_MIN, L_MAX)

print("Тестирование прокси модели (на jnd jnd jnd)")
df_3jnd = get_results_for_model(
    model=model_3jnd,
    attack_model=proxy_model,
    loader=val_loader_1000_rgb, 
    converter=converter, 
    jnd_model=jnd_model, 
    L_MIN=L_MIN, L_MAX=L_MAX, 
    layers="jnd_jnd_jnd"
)

Тестирование прокси модели (на jnd jnd jnd)


Тестирование режима jnd_jnd_jnd:   0%|          | 0/4 [00:00<?, ?it/s]

Атака FGSM:   0%|          | 0/63 [00:00<?, ?it/s]

Атака BIM:   0%|          | 0/63 [00:00<?, ?it/s]

Атака PGD:   0%|          | 0/63 [00:00<?, ?it/s]

Атака CW:   0%|          | 0/63 [00:00<?, ?it/s]

In [47]:
display(df_3jnd)

,Clean Acc,Adv Acc,Drop,Clean Conf,Adv Conf
Attack,,,,,
FGSM,74.2,49.0,25.2,0.732492,0.482893
BIM,74.2,48.3,25.9,0.732492,0.472485
PGD,74.2,52.0,22.2,0.732492,0.510094
CW,74.2,68.6,5.6,0.732492,0.664180
